# Examen Practico 2

|                |   |
:----------------|---|
| **Nombre**     |  Jorge Oviedo Magaña |
| **Fecha**      | 16/04/2026 |
| **Expediente** |  757048 | 

## Optimización Bayesiana para encontrar minimo global en función

 

Explica por qué optimización bayesiana es una elección buena para este problema en lugar de GridSearch.
* R: Porque con Optimización Bayesiana tu escoges los valores iniciales de "x","y" y "z" en este caso estan acotadas en [0,1]

In [5]:
import numpy as np

def f(X):
    x, y, z = X
    return ((6*x-2)**2 * np.sin(12*x-4) + (4*y-3)**2 * np.cos(9*y-1) + (5*z-1)**2 * np.sin(8*z-4))

from skopt.space import Real
# Define the search space
space = [Real(0, 1, name='x'), Real(0, 1, name='y'), Real(0, 1, name='z')]

# Run Bayesian optimization (5 muestras aleatorias e iterar 15 veces)
from skopt import gp_minimize
result = gp_minimize(f, space, n_calls=15, n_random_starts=5)

print("Candidato minimo global: ", result.fun)


Candidato minimo global:  -12.318143842398115


# DATASET ADIDAS

In [6]:
import pandas as pd

df = pd.read_csv('adidas.csv')
print(df.head())

                                                 url  \
0  https://www.adidas.com/us/beach-shorts/FJ5089....   
1  https://www.adidas.com/us/five-ten-kestrel-lac...   
2  https://www.adidas.com/us/mexico-away-jersey/G...   
3  https://www.adidas.com/us/five-ten-hiangle-pro...   
4  https://www.adidas.com/us/mesh-broken-stripe-p...   

                                              name     sku  selling_price  \
0                                     Beach Shorts  FJ5089             40   
1        Five Ten Kestrel Lace Mountain Bike Shoes  BC0770            150   
2                               Mexico Away Jersey  GC7946             70   
3  Five Ten Hiangle Pro Competition Climbing Shoes  FV4744            160   
4                    Mesh Broken-Stripe Polo Shirt  GM0239             65   

  original_price currency availability  color  category                source  \
0            NaN      USD      InStock  Black  Clothing  adidas United States   
1            NaN      USD      InStock

In [ ]:
#EDA
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 845 entries, 0 to 844
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   url             845 non-null    object 
 1   name            845 non-null    object 
 2   sku             845 non-null    object 
 3   selling_price   845 non-null    int64  
 4   original_price  829 non-null    object 
 5   currency        845 non-null    object 
 6   availability    845 non-null    object 
 7   color           845 non-null    object 
 8   category        845 non-null    object 
 9   source          845 non-null    object 
 10  source_website  845 non-null    object 
 11  breadcrumbs     845 non-null    object 
 12  description     845 non-null    object 
 13  brand           845 non-null    object 
 14  images          845 non-null    object 
 15  country         845 non-null    object 
 16  language        845 non-null    object 
 17  average_rating  845 non-null    flo

### DATOS
* url (cat) -> eliminar
* name (cat)
* sku (cat)
* currency (cat)
* availability (cat)
* color (cat)
* category (cat)
* source (cat)
* source_website (cat)
* breadcrumbs (cat)
* description (cat)
* brand (cat)
* images (cat)
* country (cat)
* language (cat)
* crawled_at (cat)

* original_price (cat -> int)
* selling_price (int)
* reviews_count (int)

* average_rating (Decimals)


In [35]:
#valores unicos original_price
print(df['original_price'].unique())


#eliminar nulos
df = df.dropna()


[ 25.  80. 130.  70.  14.  35.  55.  30. 120.  43.  50. 150.  60.  85.
  75.  45.  28.  65. 100. 110.  26.  18.  40.  15.  22. 180.  20.  23.
 160. 140.  58.  90.  16.  33. 250. 280.  32. 200. 300.  52.  24.  44.]


In [36]:
y = df['average_rating']
x = df.drop(columns=['average_rating','original_price'])

#preprocessor y pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

cat_cols = x.select_dtypes(include=['object']).columns 
num_cols = x.select_dtypes(include=['int64', 'float64']).columns
model = LogisticRegression()

preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), num_cols), 
                  ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)])
from sklearn.pipeline import Pipeline

pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('model', model)])

In [ ]:
# train test split 
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
# fit model
pipeline.fit(X_train, y_train)

#cross validation
from sklearn.model_selection import cross_val_score, train_test_split
cv_scores = cross_val_score(pipeline, x, y, cv=2, scoring='accuracy')

print("Cross-validation scores: ", cv_scores)


ValueError: Unknown label type: continuous. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.

# DATASET DIABETES

In [38]:
import pandas as pd

df = pd.read_csv('diabetes.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [ ]:
X = df.drop(columns=['Outcome'])
y = df['Outcome']

num_cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'outcome', 'Age']
float_cols = ['BMI', 'DiabetesPedigreeFunction']

preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), num_cols),
                  ('float', StandardScaler(), float_cols)])

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

model = SVC(kernel='rbf', C=1, gamma='scale')

pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('model', model)])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipeline.fit(X_train, y_train)

cv_scores = cross_val_score(pipeline, X, y, cv=2, scoring='F1')
print("Cross-validation scores: ", cv_scores)


ValueError: A given column is not a column of the dataframe